In [1]:
import torch
import gc
import wandb
import warnings
import optuna

import torch.nn.functional as F
import torch.nn as nn
import numpy as np
import pandas as pd

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import ndcg_score
from torch.optim.lr_scheduler import StepLR

C:\Users\roans\anaconda3.0\envs\transformers\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
def ranking_collate_fn(batch):
    """
    Takes a list of candidate dictionaries (where each dict contains N items)
    and concatenates them along the 0th dimension to mimic PyG batching.
    """
    return {
        'cv_input_ids': torch.cat([b['cv_input_ids'] for b in batch], dim=0),
        'cv_attention_mask': torch.cat([b['cv_attention_mask'] for b in batch], dim=0),
        'vac_input_ids': torch.cat([b['vac_input_ids'] for b in batch], dim=0),
        'vac_attention_mask': torch.cat([b['vac_attention_mask'] for b in batch], dim=0),
        'labels': torch.cat([b['labels'] for b in batch], dim=0),
        
        # Flatten the nested lists of strings
        'cvid': [c for b in batch for c in b['cvid']],
        'vacancy_id': [v for b in batch for v in b['vacancy_id']]
    }

class TextRankingDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512):
        print("Pre-tokenizing dataset (this takes a minute)...")
        
        cv_texts = dataframe['cv_text'].astype(str).tolist()
        vac_texts = dataframe['vacancy_text'].astype(str).tolist()
        
        # Tokenize everything upfront
        self.cv_encodings = tokenizer(
            cv_texts, add_special_tokens=True, 
            max_length=max_length, padding='max_length', 
            truncation=True, return_tensors='pt'
        )
        self.vac_encodings = tokenizer(
            vac_texts, add_special_tokens=True, 
            max_length=max_length, padding='max_length', 
            truncation=True, return_tensors='pt'
        )

        self.labels = dataframe['response'].values
        self.cvids = dataframe['cvid'].astype(str).tolist()
        self.vacancy_ids = dataframe['humanjobid'].astype(str).tolist()
        
        self.unique_cvids = dataframe['cvid'].unique().tolist()
        self.grouped_indices = dataframe.groupby('cvid').indices

    def __len__(self):
        # Length is now the number of unique candidates, NOT the number of total rows
        return len(self.unique_cvids)

    def __getitem__(self, idx):
        # 1. Look up the candidate
        cvid = self.unique_cvids[idx]
        # 2. Get the specific row indices for all N of their vacancies
        indices = self.grouped_indices[cvid] 
        
        # 3. Return the N-sized chunk of tensors for this single candidate
        return {
            'cv_input_ids': self.cv_encodings['input_ids'][indices],
            'cv_attention_mask': self.cv_encodings['attention_mask'][indices],
            'vac_input_ids': self.vac_encodings['input_ids'][indices],
            'vac_attention_mask': self.vac_encodings['attention_mask'][indices],
            'labels': torch.tensor(self.labels[indices], dtype=torch.float32),
            'cvid': [self.cvids[i] for i in indices],
            'vacancy_id': [self.vacancy_ids[i] for i in indices]
        }

In [3]:
trainloader = torch.load(f'../dataloaders/text_trainloader.pth',
                         weights_only=False)
valloader = torch.load(f'../dataloaders/text_valloader.pth',
                         weights_only=False)
testloader = torch.load(f'../dataloaders/text_testloader.pth',
                         weights_only=False)

In [4]:
def listwise_loss(scores, labels):
    """
    Compute the LambdaRank loss. (assume sigma=1.)
    """
    if labels.size(0) < 2:
        return torch.zeros_like(scores)

    N = torch.arange(len(scores))
    num_docs = len(scores)
    sigma = 1

    # Calculate lambda_{i, j} for every <i, j>.
    S_j = torch.stack([labels] * num_docs)
    S_i = S_j.T

    S = torch.nan_to_num((S_i - S_j) / (S_i - S_j).abs())
    lamda = (sigma * (0.5 * (1 - S) - (1 / (1 + torch.exp(sigma * (scores - scores.T))))))

    # Calculate abs(Delta-NDCG) for each ordering <i, j> combination
    sorted_ind = torch.flip(scores.argsort(dim=0).flatten(), dims=[0])
    sorted_labels = labels[sorted_ind]
    ideal_labels = torch.sort(labels)[0].flip(dims=[0])
    k = (torch.arange(sorted_labels.shape[0]) + 1).to(scores.device)
    
    DCG_ideal_labels = torch.sum((2**ideal_labels - 1) / torch.log(k + 1)) 
    
    # --- THE FIX: Prevent Division by Zero ---
    # If all labels are 0, ideal DCG is 0. There's no ranking to learn.
    if DCG_ideal_labels == 0:
        return torch.zeros_like(scores)
    # -----------------------------------------

    doc_id_to_rank = torch.Tensor([(sorted_ind == i).nonzero(as_tuple=True)[0] for i in N]).int()
    doc_id_to_label = torch.Tensor([sorted_labels[R_i] for R_i in doc_id_to_rank]).int().to(scores.device)
    
    # Calculate delta NDCG
    R_j = torch.stack([doc_id_to_rank] * num_docs).to(scores.device)
    R_i = R_j.T
    label_j = torch.stack([doc_id_to_label] * num_docs).to(scores.device)
    label_i = label_j.T
    DCG_discount = ((2**label_i - 1) / torch.log(R_i + 2) + (2**label_j - 1) / torch.log(R_j + 2)).to(scores.device)
    DCG_gain = ((2**label_j - 1) / torch.log(R_i + 2) + (2**label_i - 1) / torch.log(R_j + 2)).to(scores.device)
    delta_NDCG = ((DCG_gain - DCG_discount) / DCG_ideal_labels).abs()

    lambda_rank_loss =  (lamda * delta_NDCG).sum(axis=1).unsqueeze(1) 

    return lambda_rank_loss

In [5]:
class text_ranker(torch.nn.Module):
    def __init__(self, pooling="mean"):
        super().__init__()
        self.model = AutoModel.from_pretrained("jjzha/dajobbert-base-uncased")
        self.pooling = pooling
        
        for name, param in self.model.named_parameters():
            if 'encoder.layer' in name:
                layer_num = int(name.split('.')[2])
                if layer_num < 8:  # Freezes layers 0 through 7
                    param.requires_grad = False

    def pool_embeddings(self, outputs, attention_mask):
        """Helper function to cleanly pool the token embeddings"""
        if self.pooling == "mean":
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(outputs.last_hidden_state.size())
            sum_embeddings = torch.sum(outputs.last_hidden_state * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9) 
            return sum_embeddings / sum_mask
        elif self.pooling == "sum":
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(outputs.last_hidden_state.size())
            return torch.sum(outputs.last_hidden_state * input_mask_expanded, 1)
        elif self.pooling == "max":
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(outputs.last_hidden_state.size()).bool()
            masked_embeddings = outputs.last_hidden_state * input_mask_expanded 
            embeddings, _ = torch.max(masked_embeddings, dim=1) 
            return embeddings

    def forward(self, batch_cv, batch_vac):
        # 1. Embed the CV
        cv_outputs = self.model(batch_cv['input_ids'], attention_mask=batch_cv['attention_mask'])
        cv_emb = self.pool_embeddings(cv_outputs, batch_cv['attention_mask'])
        
        # 2. Embed the Vacancy
        vac_outputs = self.model(batch_vac['input_ids'], attention_mask=batch_vac['attention_mask'])
        vac_emb = self.pool_embeddings(vac_outputs, batch_vac['attention_mask'])
        
        # 3. Calculate Cosine Similarity 
        # F.cosine_similarity returns [batch_size], we unsqueeze to [batch_size, 1] for listwise_loss
        scores = F.cosine_similarity(cv_emb, vac_emb)
        return scores.unsqueeze(1)

In [6]:
df_gender = pd.read_csv("anonid_gender_mapping.csv")
gender_map = dict(zip(df_gender['anon_id'], df_gender['gender']))

df_location = pd.read_csv("job_area_mapping.csv")
area_map = dict(zip(df_location["humanjobid"], df_location["area"]))

In [7]:
def calculate_area_disparate_visibility(y_pred, vac_ids, area_map):
    """
    Calculates the ratio of average visibility between Rural and Urban vacancies.
    """
    scores = y_pred.detach().view(-1).cpu().numpy()
    sorted_indices = np.argsort(-scores)
    
    exposures = {'Urban': [], 'Rural': [], "Unknown": []}
    
    for rank_0_idx, orig_idx in enumerate(sorted_indices):
        rank = rank_0_idx + 1  # 1-based indexing
        
        # Safely extract vacancy id
        vac_id = vac_ids[orig_idx]
        vac_id = vac_id.item() if hasattr(vac_id, 'item') else vac_id
        
        area = area_map.get(int(float(vac_id)), "Unknown")
        
        if area in exposures:
            exposure = 1.0 / np.log2(1.0 + rank)
            exposures[area].append(exposure)
            
    avg_urban_vis = np.mean(exposures['Urban']) if exposures['Urban'] else 0.0
    avg_rural_vis = np.mean(exposures['Rural']) if exposures['Rural'] else 0.0
    
    # Return ratio (Rural / Urban)
    if avg_urban_vis > 0 and avg_rural_vis > 0:
        return avg_rural_vis / avg_urban_vis
    return None

In [8]:
def train_loop(model, optimizer, trainloader, use_wandb=False):
    ndcg_scores = []
    scaler = torch.cuda.amp.GradScaler() 
        
    for i, batch in enumerate(trainloader):
        batch_cv = {
            'input_ids': batch['cv_input_ids'].to(device),
            'attention_mask': batch['cv_attention_mask'].to(device)
        }
        batch_vac = {
            'input_ids': batch['vac_input_ids'].to(device),
            'attention_mask': batch['vac_attention_mask'].to(device)
        }
        ground_truth = batch['labels'].to(device)
        optimizer.zero_grad()
        
        # 1. Forward Pass in Mixed Precision
        with torch.cuda.amp.autocast():
            y_pred = model(batch_cv, batch_vac)
            if len(y_pred) > len(ground_truth):
                y_pred = y_pred[:len(ground_truth)]
                
            # listwise_loss calculates the EXPLICIT gradients, not a scalar loss
            lambda_i = listwise_loss(y_pred, ground_truth)
        
        # 2. Backward Pass
        scaled_lambda_i = scaler.scale(lambda_i)
        
        # Apply the scaled custom gradients backward through the network
        torch.autograd.backward(y_pred, scaled_lambda_i)
        
        # 3. Optimizer Step
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        
        scaler.step(optimizer)
        scaler.update()

        score = ndcg_score(ground_truth.detach().cpu().unsqueeze(0), 
                           y_pred.squeeze().unsqueeze(0).detach().cpu(), k=10)
        ndcg_scores.append(score)
        
        # flush=True prevents Jupyter from indefinitely buffering the output
        print(f"Batch: {i + 1}/{len(trainloader)}, y_pred mean: {y_pred.mean():.4f}, nDCG: {score:.4f}    ", end="\r", flush=True)
        
    return ndcg_scores


def val_loop(model, valloader, gender_map, area_map):
    model.eval()
    
    # Track overall NDCG
    all_scores = []
    
    # Track NDCG separately by gender (Performance Disparity)
    ndcg_scores = {'Male': [], 'Female': [], 'Other': [], 'Unknown': []}
    
    # Track global counts for dataset-relative Disparate Visibility (ΔV)
    total_rural_dataset = 0
    total_items_dataset = 0
    total_rural_recommended = 0
    total_items_recommended = 0
    
    vacancy_pool_sizes = []
    
    with torch.no_grad():
        for i, batch in enumerate(valloader):
            print(f"Batch: {i + 1}/{len(valloader)}", end="\r")
            
            batch_cv = {
                'input_ids': batch['cv_input_ids'].to(device),
                'attention_mask': batch['cv_attention_mask'].to(device)
            }
            batch_vac = {
                'input_ids': batch['vac_input_ids'].to(device),
                'attention_mask': batch['vac_attention_mask'].to(device)
            }
            ground_truth = batch['labels'].to(device)
            
            # Forward pass
            y_pred_val = model(batch_cv, batch_vac)
                
            # Safely extract vacancy IDs before potential truncation
            vac_ids = batch.get('vacancy_id', [])
                
            if len(y_pred_val) > len(ground_truth):
                y_pred_val = y_pred_val[:len(ground_truth)]
                if len(vac_ids) > len(ground_truth):
                    vac_ids = vac_ids[:len(ground_truth)]

            vac_ids = [int(_id) for _id in vac_ids]
        
            # Format tensors for sklearn
            y_true = ground_truth.detach().cpu().unsqueeze(0)
            y_score = y_pred_val.squeeze().unsqueeze(0).detach().cpu()
            
            batch_ndcg = ndcg_score(y_true, y_score, k=10)
            all_scores.append(batch_ndcg)
            
            # --- Gender Fairness (Utility) ---
            cvid_raw = batch.get("cvid", [None])
            candidate_id = cvid_raw[0][0] if isinstance(cvid_raw[0], (list, tuple)) else cvid_raw[0]
            
            gender = gender_map.get(candidate_id, 'Unknown')
            if candidate_id is not None:
                ndcg_scores[gender].append(batch_ndcg)
            
            # --- Geographic Fairness (Dataset-Relative Disparate Visibility ΔV) ---
            vacancy_pool_sizes.append(len(vac_ids))
            
            if len(vac_ids) > 0:
                # 1. Baseline: Accumulate Rural items and total items across the dataset pool
                batch_rural_count = sum(1 for v in vac_ids if area_map.get(v) == 'Rural')
                total_rural_dataset += batch_rural_count
                total_items_dataset += len(vac_ids)
                
                # 2. Recommendations: Accumulate Rural items in the Top-K recommendations
                actual_k = min(10, len(vac_ids))
                if actual_k > 0:
                    y_pred_squeeze = y_pred_val.squeeze()
                    if y_pred_squeeze.ndim == 0:
                        y_pred_squeeze = y_pred_squeeze.unsqueeze(0)
                        
                    top_k_indices = torch.topk(y_pred_squeeze, actual_k).indices.tolist()
                    if not isinstance(top_k_indices, list):
                        top_k_indices = [top_k_indices]
                        
                    top_k_vacs = [vac_ids[idx] for idx in top_k_indices]
                    top_k_rural_count = sum(1 for v in top_k_vacs if area_map.get(v) == 'Rural')
                    
                    total_rural_recommended += top_k_rural_count
                    total_items_recommended += actual_k

    # --- Utility (Gender) Summary ---
    mean_male_ndcg = np.mean(ndcg_scores['Male']) if ndcg_scores['Male'] else 0.0
    mean_female_ndcg = np.mean(ndcg_scores['Female']) if ndcg_scores['Female'] else 0.0
    
    print(f"\nFemale NDCG: {mean_female_ndcg:.4f} | Male NDCG: {mean_male_ndcg:.4f}")

    if len(ndcg_scores['Male']) > 0 and len(ndcg_scores['Female']) > 0:
        ndcg_gap = mean_female_ndcg - mean_male_ndcg
        print(f"Performance Disparity : {ndcg_gap:.4f}\n")
    else:
        print("Performance Disparity : N/A (Missing demographic group)\n")
        ndcg_gap = None

    # --- Geographic Fairness Summary ---
    avg_pool = np.mean(vacancy_pool_sizes) if vacancy_pool_sizes else 0
    min_pool = np.min(vacancy_pool_sizes) if vacancy_pool_sizes else 0
    max_pool = np.max(vacancy_pool_sizes) if vacancy_pool_sizes else 0
    
    print("--- Geographic Fairness Summary ---")
    print(f"Total Batches (Candidates) : {len(vacancy_pool_sizes)}")
    print(f"Vacancies per Batch        : Mean: {avg_pool:.1f} | Min: {min_pool} | Max: {max_pool}")
    
    if total_items_dataset > 0 and total_items_recommended > 0:
        frac_dataset = total_rural_dataset / total_items_dataset
        frac_recommended = total_rural_recommended / total_items_recommended
        mean_disp_vis = frac_recommended - frac_dataset
        
        print(f"Rural Fraction in Dataset      : {frac_dataset * 100:.2f}%")
        print(f"Rural Fraction in Top-10 Recoms: {frac_recommended * 100:.2f}%")
        print(f"Disparate Visibility (\u0394V)      : {mean_disp_vis:.4f}\n")
    else:
        print("Disparate Visibility (\u0394V): N/A (No valid vacancy pools)\n")
        mean_disp_vis = None
            
    return all_scores, ndcg_gap, mean_disp_vis

In [9]:
def train_model(model, optimizer, scheduler, trainloader, valloader, gender_map, area_map, epochs=10, step_size=5):
    best_score = 0
    final_ndcg_gap = 0
    final_disp_vis = 0
    
    for epoch in range(epochs + 1):
        print(f"Epoch: {epoch}/{epochs}")

        # Train the model for the current epoch
        epoch_ndcg_scores = train_loop(model, optimizer, trainloader)

        print(f"\nTraining nDCG: {np.mean(epoch_ndcg_scores):.4f}\n")
        scheduler.step()

        # Evaluate the model
        val_ndcg_scores, ndcg_gap, mean_disp_vis = val_loop(model, valloader, gender_map, area_map)
        print(f"\nTesting nDCG: {np.mean(val_ndcg_scores):.4f}\n")

        if np.mean(val_ndcg_scores) > best_score:
            best_score = np.mean(val_ndcg_scores)
            final_ndcg_gap = ndcg_gap
            final_disp_vis = mean_disp_vis
            
    return best_score, final_ndcg_gap, final_disp_vis

In [10]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.cuda.empty_cache() 
gc.collect()

best_config = {'learning_rate': 8.493296693542725e-05, 'pooling_method': 'mean', 'epochs': 5}

warnings.filterwarnings('ignore')

model = text_ranker(pooling=best_config["pooling_method"]).to(device)      
model.train()
    
optimizer = torch.optim.Adam(model.parameters(), lr=best_config["learning_rate"])
scheduler = StepLR(optimizer, step_size=3, gamma=0.1)

train_model(model, optimizer, scheduler, trainloader, testloader, gender_map, area_map,
            epochs=best_config["epochs"], step_size=5)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: jjzha/dajobbert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch: 0/5
Batch: 289/289, y_pred mean: 0.9991, nDCG: 0.3869    
Training nDCG: 0.3891

Batch: 37/37
Female NDCG: 0.3948 | Male NDCG: 0.3971
Performance Disparity : 0.0023

--- Geographic Fairness Summary ---
Total Batches (Candidates) : 37
Vacancies per Batch        : Mean: 17.4 | Min: 10 | Max: 27
Rural Fraction in Dataset      : 45.89%
Rural Fraction in Top-10 Recoms: 48.92%
Disparate Visibility (ΔV)      : 0.0303


Testing nDCG: 0.4127

Epoch: 1/5
Batch: 289/289, y_pred mean: 0.9250, nDCG: 0.5641    
Training nDCG: 0.4232

Batch: 37/37
Female NDCG: 0.4419 | Male NDCG: 0.4023
Performance Disparity : -0.0396

--- Geographic Fairness Summary ---
Total Batches (Candidates) : 37
Vacancies per Batch        : Mean: 17.4 | Min: 10 | Max: 27
Rural Fraction in Dataset      : 45.89%
Rural Fraction in Top-10 Recoms: 47.84%
Disparate Visibility (ΔV)      : 0.0195


Testing nDCG: 0.4313

Epoch: 2/5
Batch: 289/289, y_pred mean: 0.9043, nDCG: 1.0000     
Training nDCG: 0.4520

Batch: 37/37
Female 

(0.4740495654277288, -0.08755749031237997, 0.02216635239891057)